# 03 Первое работающее решение
Для видео используем готовый детектор HOG+SVM из OpenCV. Рамки между кадрами сопоставляются по IoU от 0,3 с сохранением ID до трёх пропущенных кадров. Обучать модель для этой версии не требуется. Трекер запускаем на публичных детекциях MOT17-09 и MOT20-03 и измеряем ошибку подсчёта. Полученные метрики относятся к этому входу, а видеорежим HOG+SVM проверяется отдельно.


In [1]:
from collections import defaultdict
from configparser import ConfigParser
from pathlib import Path

import numpy as np
from baseline import Tracker, Queue, analyze

for dataset, directory in [("MOT17", "MOT17-09-FRCNN"), ("MOT20", "MOT20-03")]:
    scene = Path("data") / dataset / "train" / directory
    info = ConfigParser()
    info.read(scene / "seqinfo.ini")
    frames = info["Sequence"].getint("seqLength")
    truth = np.loadtxt(scene / "gt/gt.txt", delimiter=",", ndmin=2)
    truth = truth[(truth[:, 6] == 1) & (truth[:, 7] == 1)]
    counts = np.bincount(truth[:, 0].astype(int), minlength=frames + 1)[1:]
    detections = np.loadtxt(scene / "det/det.txt", delimiter=",", ndmin=2)
    by_frame = defaultdict(list)
    for row in detections:
        if np.isfinite(row[:7]).all() and row[6] >= 0 and min(row[4:6]) > 0:
            by_frame[int(row[0])].append(row[2:6])
    tracker = Tracker()
    predicted = np.array([len(tracker.update(frame, by_frame[frame]))
                          for frame in range(1, frames + 1)])
    errors = predicted - counts
    print(dataset, "кадров:", frames, "MAE:", round(np.abs(errors).mean(), 2),
          "RMSE:", round(np.sqrt((errors ** 2).mean()), 2))
    if dataset == "MOT17":
        for i in (np.abs(errors).argmin(), np.abs(errors).argmax()):
            print(f"Кадр {i + 1}: разметка {counts[i]}, прогноз {predicted[i]}")


MOT17 кадров: 525 MAE: 4.34 RMSE: 4.66
Кадр 81: разметка 7, прогноз 7
Кадр 288: разметка 13, прогноз 5


MOT20 кадров: 2405 MAE: 56.68 RMSE: 62.68


В плотной сцене MOT20 ошибка выше. MAE и RMSE измеряют ошибку в людях на кадр. Время ожидания проверяем отдельно на известной траектории, потому что в разметке MOT нет событий очереди.


In [2]:
tracker = Tracker()
first_id = tracker.update(1, [(10, 10, 30, 60)])[0][0]
assert tracker.update(2, [(15, 10, 30, 60)])[0][0] == first_id
queue = Queue((0, 0, 50, 100), (50, 0, 100, 100), fps=2, area=2)
waiting = queue.update(1, [(1, (10, 10, 10, 20))], {1})
served = queue.update(4, [(1, (60, 10, 10, 20))], {1})
assert waiting["queue_length"] == 1 and waiting["density_m2"] == .5
assert served["queue_length"] == 0 and served["completed_wait_s"] == [1.5]
print(waiting)
print(served)


{'visible': 1, 'queue_length': 1, 'density_m2': 0.5, 'completed_wait_s': []}
{'visible': 1, 'queue_length': 0, 'density_m2': 0.0, 'completed_wait_s': [1.5]}


## Запуск
```bash
.venv/bin/python baseline.py data/example.mp4 --output queue.jsonl
```

Настройки зон задаются параметрами командной строки. Порог IoU и правила сопоставления сохранены в `baseline.py`. Предобученный детектор поставляется с OpenCV. При перекрытии людей детектор пропускает рамки, а трекер может потерять ID.


In [3]:
video = Path("data/example.mp4")
if video.is_file():
    frames = analyze(str(video), (10, 80, 300, 360), (300, 80, 400, 360), None, "queue.jsonl")
    assert len(Path("queue.jsonl").read_text().splitlines()) == frames
    print("Обработано кадров:", frames)
    print(Path("queue.jsonl").read_text().splitlines()[0])
else:
    print("Для видеопроверки поместите MP4 в data/example.mp4.")


Обработано кадров: 8
{"frame": 1, "visible": 1, "queue_length": 0, "density_m2": null, "completed_wait_s": []}
